# 8. Validación estadística de hipótesis de negocio

Este notebook toma las hipótesis/KPIs planteados en `6.Hipotesis_Y_KPIs.ipynb` y los convierte en contrastes estadísticos formales cuando corresponde.

La pregunta de negocio es si los patrones relevantes para un inversor flipper tienen evidencia estadística: diferencias de precio por barrio, concentración de stock mejorable, relación entre amenities y precio, accesibilidad urbana, descuentos frente a comparables y oportunidad por cluster.

Aclaración metodológica: varios puntos del notebook 6 son KPIs descriptivos, no hipótesis inferenciales puras. En esos casos, la validación estadística evalúa si el patrón observado muestra diferencias/asociaciones estadísticamente significativas en la muestra disponible.


## Correspondencia con el notebook 6

| Punto en notebook 6 | KPI / hipotesis operativa | Validacion en este notebook |
|---|---|---|
| 1. Precio de entrada por zona | El precio por m2 cambia segun barrio | Kruskal-Wallis de `Precio_m2` por barrio |
| 2. Profundidad de mercado | La oferta no se distribuye uniformemente entre barrios | Chi-cuadrado de bondad de ajuste sobre conteos por barrio |
| 3. Stock refaccionable/mejorable | La proporcion de propiedades mejorables cambia por barrio | Chi-cuadrado de independencia barrio vs estado mejorable |
| 4. Amenities por segmento de precio | Los amenities se asocian al segmento/precio | Spearman entre cantidad de amenities y `Precio_m2` |
| 5. Accesibilidad urbana por barrio | La accesibilidad urbana cambia segun barrio | Kruskal-Wallis de indice de accesibilidad por barrio |
| 6. Descuento frente a comparables | La subvaluacion fuerte no aparece igual en todos los barrios | Chi-cuadrado barrio vs descuento >= 15% |
| 7. Prima por dotacion de amenities | El precio por m2 cambia segun dotacion de amenities | Kruskal-Wallis por dotacion baja/media/alta |
| 8. Accesibilidad y precio por m2 | El precio por m2 cambia segun rango de accesibilidad | Kruskal-Wallis por cuartil de accesibilidad |
| 9. Distancia al subte | El precio por m2 cambia segun distancia al subte | Kruskal-Wallis por rango de distancia al subte |
| 10. Indice de oportunidad de flip | El score de oportunidad cambia segun cluster/barrio | Kruskal-Wallis del indice de oportunidad por cluster |


## Criterio de decision

- Nivel de significancia: $\alpha = 0.05$.
- Se priorizan pruebas no parametricas porque `Precio_m2` es asimetrico y sensible a outliers.
- Se corrigen los p-values con Holm-Bonferroni por multiples contrastes.
- Se reporta tamaño de efecto para distinguir significancia estadistica de relevancia practica.

In [1]:
import re
import unicodedata

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ALPHA = 0.05

df = pd.read_csv("../data/processed/Argenprop_limpio.csv", encoding="utf-8-sig")
print(f"Dataset utilizado: {df.shape[0]:,} filas y {df.shape[1]:,} variables")
df.head()

Dataset utilizado: 7,245 filas y 55 variables


,sintetica_id_registro,imputada_Precio,imputada_Expensas,original_Calle,original_Altura,imputada_Piso,original_Link,imputada_Ambientes,imputada_Dormitorios,original_Banos,imputada_Estado,imputada_Antiguedad,imputada_Disposicion,imputada_Tipo_Unidad,imputada_Sup_Cubierta_m2,imputada_Sup_Total_m2,original_Aire_acondicionado_individual,original_Losa_radiante,original_Gas_natural,original_Agua_corriente,original_Balcon,original_Terraza,original_Jardin,original_Patio,original_Baulera,original_Cochera,original_Muebles_de_cocina,original_Permite_Mascotas,original_Ascensor,original_Pileta,original_Parrilla,original_Gimnasio,original_Sauna,original_Laundry,original_Vigilancia,enriquecida_Latitud,enriquecida_Longitud,enriquecida_Barrio,enriquecida_Comuna,enriquecida_Dist_Subte_m,enriquecida_Subte_cercano,enriquecida_Linea_subte,enriquecida_Dist_Hospital_m,enriquecida_Hospital_cercano,enriquecida_Dist_Colegio_m,enriquecida_Colegios_500m,enriquecida_Dist_Comisaria_m,enriquecida_Dist_Gimnasio_m,enriquecida_Dist_Supermercado_m,enriquecida_Supermercados_500m,enriquecida_Dist_Avenida_m,enriquecida_Avenida_cercana,enriquecida_Paradas_colectivo_300m,imputada_Antiguedad_imputada,sintetica_Cluster
0,1,"150,000.0000","260,000.0000",Bulnes,"1,600.0000",NaN,https://www.argenprop.com/departamento-en-vent...,3,2,1.0000,Excelente,30,Frente,Departamento,60.0000,60.0000,1,0,1,1,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,-34.5868,-58.4107,Palermo,14,167.7761,BULNES,D,719.1936,Dr. J. A. Fernandez,180.3531,10,493.1267,51.7275,20.2793,10,156.7883,Avenida Santa Fe,19,0,2
1,2,"330,000.0000","203,300.0000",ARAOZ,"1,200.0000",8.0000,https://www.argenprop.com/departamento-en-vent...,4,3,2.0000,Bueno,50,Contrafrente,No disponible,90.0000,96.0000,0,0,0,0,1,1,0,0,1,0,0,1,0,0,1,0,0,0,0,-34.5889,-58.4202,Palermo,14,563.7326,R.SCALABRINI ORTIZ,D,948.6219,R. Gutierrez,188.5377,11,403.1148,429.5844,249.2481,4,96.5150,Avenida Raúl Scalabrini Ortiz,10,1,2
2,3,"270,000.0000","300,000.0000",Honduras,"3,900.0000",2.0000,https://www.argenprop.com/departamento-en-vent...,4,2,2.0000,Excelente,20,Frente,Semipiso,87.0000,87.0000,1,0,1,1,1,1,0,0,0,1,1,0,1,0,0,0,0,0,0,-34.5813,-58.4400,Palermo,14,815.4656,MINISTRO CARRANZA - MIGUEL ABUELO,D,"2,930.4049",Hospital Municipal de Oncologia M. Curie,240.8739,7,891.1453,383.0179,75.2437,7,425.4954,Avenida Coronel Niceto Vega,5,0,2
3,4,"570,000.0000","1,000,000.0000",Castex,"3,300.0000",NaN,https://www.argenprop.com/departamento-en-vent...,4,3,3.0000,No disponible,40,No disponible,Piso,140.0000,160.0000,0,1,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,-34.5761,-58.4076,Palermo,14,"1,262.2533",R.SCALABRINI ORTIZ,D,559.8688,Dr. J. A. Fernandez,256.7976,4,804.1383,286.6980,217.0756,3,108.4441,Avenida Casares,7,0,2
4,5,"98,000.0000","150,000.0000",GURRUCHAGA,"2,100.0000",5.0000,https://www.argenprop.com/departamento-en-vent...,1,1,1.0000,Muy Bueno,15,Frente,No disponible,31.0000,34.0000,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,-34.5883,-58.4277,Palermo,14,973.2733,PLAZA ITALIA,D,"1,574.5700",R. Gutierrez,153.6102,5,461.4835,393.5680,85.2867,6,378.8029,Avenida Raúl Scalabrini Ortiz,7,0,2


## Compatibilidad con variables prefijadas

El limpio final tiene nombres como `imputada_Precio`, `enriquecida_Barrio` y `sintetica_Cluster`. Esta celda arma un mapa para trabajar con nombres conceptuales y no depender del prefijo.

In [2]:
def normalizar_nombre(nombre):
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = nombre.encode("ascii", "ignore").decode("ascii")
    nombre = re.sub(r"[^0-9A-Za-z_]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre).strip("_")
    return nombre

columnas_por_base = {}
for columna in df.columns:
    partes = str(columna).split("_", 1)
    base = partes[1] if partes[0] in {"original", "imputada", "sintetica", "enriquecida"} and len(partes) > 1 else str(columna)
    columnas_por_base[normalizar_nombre(base)] = columna

def c(nombre):
    key = normalizar_nombre(nombre)
    if key not in columnas_por_base:
        raise KeyError(f"No se encontro la columna conceptual {nombre!r}. Columnas disponibles: {list(df.columns)[:10]}...")
    return columnas_por_base[key]

variables_clave = [
    "Precio", "Expensas", "Sup_Cubierta_m2", "Sup_Total_m2", "Ambientes",
    "Dormitorios", "Banos", "Antiguedad", "Barrio", "Estado", "Tipo_Unidad",
    "Dist_Subte_m", "Paradas_colectivo_300m", "Cluster"
]

{nombre: c(nombre) for nombre in variables_clave}

{'Precio': 'imputada_Precio',
 'Expensas': 'imputada_Expensas',
 'Sup_Cubierta_m2': 'imputada_Sup_Cubierta_m2',
 'Sup_Total_m2': 'imputada_Sup_Total_m2',
 'Ambientes': 'imputada_Ambientes',
 'Dormitorios': 'imputada_Dormitorios',
 'Banos': 'original_Banos',
 'Antiguedad': 'imputada_Antiguedad',
 'Barrio': 'enriquecida_Barrio',
 'Estado': 'imputada_Estado',
 'Tipo_Unidad': 'imputada_Tipo_Unidad',
 'Dist_Subte_m': 'enriquecida_Dist_Subte_m',
 'Paradas_colectivo_300m': 'enriquecida_Paradas_colectivo_300m',
 'Cluster': 'sintetica_Cluster'}

## Preparacion de variables de analisis

Se reconstruyen las mismas variables analiticas del notebook 6: precio por m2, cantidad de amenities, indice de accesibilidad, rangos, subvaluacion frente a comparables e indice de oportunidad de flip.

In [3]:
work = df.copy()

work = work[(work[c("Precio")].notna()) & (work[c("Sup_Total_m2")].notna()) & (work[c("Sup_Total_m2")] > 0)].copy()
work["Precio_m2"] = work[c("Precio")] / work[c("Sup_Total_m2")]

amenities_base = [
    "Aire_acondicionado_individual", "Losa_radiante", "Gas_natural", "Agua_corriente",
    "Balcon", "Terraza", "Jardin", "Patio", "Baulera", "Cochera",
    "Muebles_de_cocina", "Permite_Mascotas", "Ascensor", "Pileta", "Parrilla",
    "Gimnasio", "Sauna", "Laundry", "Vigilancia"
]
amenities = [c(nombre) for nombre in amenities_base if normalizar_nombre(nombre) in columnas_por_base]
work["Cantidad_amenities"] = work[amenities].sum(axis=1)
work["Rango_precio"] = pd.qcut(work[c("Precio")], q=4, labels=["Bajo", "Medio-bajo", "Medio-alto", "Alto"], duplicates="drop")

def inv_score(series):
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.quantile(0.01), s.quantile(0.99)
    clipped = s.clip(lo, hi)
    denom = hi - lo
    if pd.isna(denom) or denom == 0:
        return pd.Series(0, index=s.index)
    return (1 - (clipped - lo) / denom).clip(0, 1).fillna(0)

work["score_subte"] = inv_score(work[c("Dist_Subte_m")])
work["score_colectivos"] = (work[c("Paradas_colectivo_300m")] / work[c("Paradas_colectivo_300m")].quantile(0.99)).clip(0, 1).fillna(0)
work["score_avenida"] = inv_score(work[c("Dist_Avenida_m")])
work["score_hospital"] = inv_score(work[c("Dist_Hospital_m")])
work["score_colegio"] = inv_score(work[c("Dist_Colegio_m")])
work["score_supermercado"] = inv_score(work[c("Dist_Supermercado_m")])

work["Indice_accesibilidad"] = (
    0.25 * work["score_subte"] +
    0.20 * work["score_colectivos"] +
    0.20 * work["score_avenida"] +
    0.15 * work["score_hospital"] +
    0.10 * work["score_colegio"] +
    0.10 * work["score_supermercado"]
)

work["Estado_mejorable"] = work[c("Estado")].isin(["A Refaccionar", "Regular", "Bueno"]).astype(int)
work["Dotacion_amenities"] = pd.qcut(work["Cantidad_amenities"].rank(method="first"), q=3, labels=["Baja", "Media", "Alta"])
work["Rango_accesibilidad"] = pd.qcut(work["Indice_accesibilidad"].rank(method="first"), q=4, labels=["Baja", "Media-baja", "Media-alta", "Alta"])
work["Rango_dist_subte"] = pd.cut(
    work[c("Dist_Subte_m")],
    bins=[0, 250, 500, 750, 1000, np.inf],
    labels=["0-250 m", "250-500 m", "500-750 m", "750-1000 m", "Mas de 1000 m"],
    include_lowest=True
)

comparables = work.groupby([c("Barrio"), c("Tipo_Unidad"), c("Ambientes")])["Precio_m2"].transform("median")
conteo_comparables = work.groupby([c("Barrio"), c("Tipo_Unidad"), c("Ambientes")])["Precio_m2"].transform("size")
work["Precio_m2_comparable"] = comparables
work["N_comparables"] = conteo_comparables
work["Subvaluacion_%"] = ((work["Precio_m2_comparable"] - work["Precio_m2"]) / work["Precio_m2_comparable"] * 100).replace([np.inf, -np.inf], np.nan)
work["Descuento_fuerte"] = (work["Subvaluacion_%"] >= 15).astype(int)

potencial_zona = work.groupby(c("Barrio"))["Precio_m2"].transform("median")
work["Potencial_zona"] = pd.qcut(potencial_zona.rank(method="first"), q=4, labels=False, duplicates="drop") / 3

subvaluacion_pos = work["Subvaluacion_%"].clip(lower=0)
work["score_subvaluacion"] = (subvaluacion_pos / subvaluacion_pos.quantile(0.99)).clip(0, 1).fillna(0)
densidad_barrio = work.groupby(c("Barrio"))[c("Precio")].transform("size")
work["score_profundidad"] = (densidad_barrio / densidad_barrio.quantile(0.99)).clip(0, 1).fillna(0)

work["Indice_oportunidad_flip"] = 100 * (
    0.40 * work["score_subvaluacion"] +
    0.20 * work["Indice_accesibilidad"] +
    0.15 * work["Potencial_zona"].fillna(0) +
    0.15 * work["Estado_mejorable"] +
    0.10 * work["score_profundidad"]
)

work[["Precio_m2", "Cantidad_amenities", "Indice_accesibilidad", "Subvaluacion_%", "Indice_oportunidad_flip"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Precio_m2,"7,245.0000","2,292.1986",879.2487,465.1163,"1,703.5714","2,160.0000","2,714.2857","18,888.8889"
Cantidad_amenities,"7,245.0000",3.1165,2.3197,0.0000,1.0000,3.0000,4.0000,14.0000
Indice_accesibilidad,"7,245.0000",0.6684,0.1296,0.1453,0.5980,0.6873,0.7605,0.9367
Subvaluacion_%,"7,245.0000",-4.5302,32.2085,-960.2940,-16.9143,0.0000,13.2757,86.0316
Indice_oportunidad_flip,"7,245.0000",33.6152,13.9118,3.1139,23.6667,31.6359,40.5491,96.1210


## Diagnostico de normalidad

Se evalua `Precio_m2` para justificar el uso de pruebas no parametricas.

In [4]:
precio_m2 = work["Precio_m2"].replace([np.inf, -np.inf], np.nan).dropna()
normal_stat, normal_p = stats.normaltest(precio_m2)

normalidad = pd.DataFrame([
    {
        "variable": "Precio_m2",
        "n": len(precio_m2),
        "test": "D'Agostino-Pearson",
        "estadistico": normal_stat,
        "p_value": normal_p,
        "asimetria": stats.skew(precio_m2, nan_policy="omit"),
        "curtosis": stats.kurtosis(precio_m2, nan_policy="omit"),
        "decision": "No normal" if normal_p < ALPHA else "No se rechaza normalidad"
    }
])

normalidad

,variable,n,test,estadistico,p_value,asimetria,curtosis,decision
0,Precio_m2,7245,D'Agostino-Pearson,"4,134.6217",0.0000,2.1616,20.6379,No normal


## Tests estadisticos

In [5]:
def holm_bonferroni(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m)
    running_max = 0
    for rank, idx in enumerate(order):
        adj = (m - rank) * p_values[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1)
    return adjusted

def decision(p_value, alpha=ALPHA):
    return "Se rechaza H0" if p_value < alpha else "No se rechaza H0"

def kruskal_by_group(data, group, target, min_n=30):
    temp = data[[group, target]].replace([np.inf, -np.inf], np.nan).dropna()
    counts = temp[group].value_counts()
    levels = counts[counts >= min_n].index
    temp = temp[temp[group].isin(levels)]
    groups = [g[target].to_numpy() for _, g in temp.groupby(group, observed=False)]
    stat, p_value = stats.kruskal(*groups)
    k = len(groups)
    n = len(temp)
    epsilon_squared = max((stat - k + 1) / (n - k), 0) if n > k else np.nan
    return stat, p_value, epsilon_squared, n, k

def spearman_test(data, x, y):
    temp = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    stat, p_value = stats.spearmanr(temp[x], temp[y])
    return stat, p_value, abs(stat), len(temp), 2

resultados = []

def agregar(kpi, h0, h1, test, stat, p_value, efecto, medida_efecto, n, grupos, lectura):
    resultados.append({
        "kpi_notebook_6": kpi,
        "H0": h0,
        "H1": h1,
        "test": test,
        "estadistico": stat,
        "p_value": p_value,
        "tamano_efecto": efecto,
        "medida_efecto": medida_efecto,
        "n": n,
        "grupos": grupos,
        "lectura": lectura
    })

# 1. Precio de entrada por zona
stat, p, efecto, n, k = kruskal_by_group(work, c("Barrio"), "Precio_m2", min_n=30)
agregar("1. Precio de entrada por zona", "Precio_m2 tiene distribucion similar entre barrios", "Al menos un barrio difiere en Precio_m2", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida que el barrio es una dimension relevante para comparar precios por m2.")

# 2. Profundidad de mercado
conteos_barrio = work[c("Barrio")].value_counts()
stat, p = stats.chisquare(conteos_barrio.values)
efecto = np.sqrt(stat / (conteos_barrio.sum() * (len(conteos_barrio) - 1)))
agregar("2. Profundidad de mercado", "La oferta se distribuye uniformemente entre barrios", "La oferta no se distribuye uniformemente entre barrios", "Chi-cuadrado bondad de ajuste", stat, p, efecto, "w", int(conteos_barrio.sum()), len(conteos_barrio), "Valida que la profundidad de mercado varia significativamente por barrio.")

# 3. Stock refaccionable/mejorable
tabla_estado = pd.crosstab(work[c("Barrio")], work["Estado_mejorable"])
stat, p, _, _ = stats.chi2_contingency(tabla_estado)
n_tabla = tabla_estado.to_numpy().sum()
cramers_v = np.sqrt(stat / (n_tabla * (min(tabla_estado.shape) - 1)))
agregar("3. Stock mejorable por barrio", "Barrio y estado mejorable son independientes", "La proporcion de mejorables depende del barrio", "Chi-cuadrado independencia", stat, p, cramers_v, "Cramers V", int(n_tabla), tabla_estado.shape[0], "Valida que el stock mejorable no aparece igual en todos los barrios.")

# 4. Amenities por segmento de precio
stat, p, efecto, n, k = spearman_test(work, "Cantidad_amenities", "Precio_m2")
agregar("4. Amenities por segmento de precio", "Cantidad de amenities y Precio_m2 no estan asociados", "Cantidad de amenities y Precio_m2 estan asociados", "Spearman", stat, p, efecto, "|rho|", n, k, "Valida si los amenities funcionan como atributo diferenciador de valor.")

# 5. Accesibilidad urbana por barrio
stat, p, efecto, n, k = kruskal_by_group(work, c("Barrio"), "Indice_accesibilidad", min_n=30)
agregar("5. Accesibilidad urbana por barrio", "El indice de accesibilidad es similar entre barrios", "Al menos un barrio difiere en accesibilidad", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida que la accesibilidad es un rasgo territorial diferenciado.")

# 6. Descuento frente a comparables
comparables_validos = work[work["N_comparables"] >= 5].copy()
tabla_descuento = pd.crosstab(comparables_validos[c("Barrio")], comparables_validos["Descuento_fuerte"])
tabla_descuento = tabla_descuento.loc[tabla_descuento.sum(axis=1) >= 30]
stat, p, _, _ = stats.chi2_contingency(tabla_descuento)
n_tabla = tabla_descuento.to_numpy().sum()
cramers_v = np.sqrt(stat / (n_tabla * (min(tabla_descuento.shape) - 1)))
agregar("6. Descuento frente a comparables", "El descuento fuerte es independiente del barrio", "La frecuencia de descuento fuerte depende del barrio", "Chi-cuadrado independencia", stat, p, cramers_v, "Cramers V", int(n_tabla), tabla_descuento.shape[0], "Valida si las oportunidades relativas se concentran mas en algunos barrios.")

# 7. Prima por dotacion de amenities
stat, p, efecto, n, k = kruskal_by_group(work, "Dotacion_amenities", "Precio_m2", min_n=30)
agregar("7. Prima por dotacion de amenities", "Precio_m2 es similar entre dotaciones de amenities", "Precio_m2 difiere segun dotacion de amenities", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida la existencia de prima de precio asociada a mayor dotacion.")

# 8. Accesibilidad y precio por m2
stat, p, efecto, n, k = kruskal_by_group(work, "Rango_accesibilidad", "Precio_m2", min_n=30)
agregar("8. Accesibilidad y precio por m2", "Precio_m2 es similar entre rangos de accesibilidad", "Precio_m2 difiere segun rango de accesibilidad", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida si la accesibilidad segmenta valores de precio por m2.")

# 9. Distancia al subte
stat, p, efecto, n, k = kruskal_by_group(work, "Rango_dist_subte", "Precio_m2", min_n=30)
agregar("9. Distancia al subte", "Precio_m2 es similar entre rangos de distancia al subte", "Precio_m2 difiere segun distancia al subte", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida si la cercania/lejana al subte cambia la distribucion de precio por m2.")

# 10. Indice de oportunidad de flip
stat, p, efecto, n, k = kruskal_by_group(work, c("Cluster"), "Indice_oportunidad_flip", min_n=30)
agregar("10. Indice de oportunidad de flip", "El indice de oportunidad es similar entre clusters", "El indice de oportunidad difiere entre clusters", "Kruskal-Wallis", stat, p, efecto, "epsilon^2", n, k, "Valida si los clusters agregados en el limpio separan perfiles de oportunidad.")

resultados = pd.DataFrame(resultados)
resultados["p_value_ajustado_holm"] = holm_bonferroni(resultados["p_value"])
resultados["decision"] = resultados["p_value_ajustado_holm"].apply(decision)

resultados[["kpi_notebook_6", "test", "estadistico", "p_value", "p_value_ajustado_holm", "decision", "tamano_efecto", "medida_efecto", "n", "grupos"]]

,kpi_notebook_6,test,estadistico,p_value,p_value_ajustado_holm,decision,tamano_efecto,medida_efecto,n,grupos
0,1. Precio de entrada por zona,Kruskal-Wallis,"2,212.9734",0.0000,0.0000,Se rechaza H0,0.3048,epsilon^2,7164,44
1,2. Profundidad de mercado,Chi-cuadrado bondad de ajuste,"6,095.7867",0.0000,0.0000,Se rechaza H0,0.1338,w,7245,48
2,3. Stock mejorable por barrio,Chi-cuadrado independencia,121.1046,0.0000,0.0000,Se rechaza H0,0.1293,Cramers V,7245,48
3,4. Amenities por segmento de precio,Spearman,0.1923,0.0000,0.0000,Se rechaza H0,0.1923,|rho|,7245,2
4,5. Accesibilidad urbana por barrio,Kruskal-Wallis,"4,130.1507",0.0000,0.0000,Se rechaza H0,0.5740,epsilon^2,7164,44
5,6. Descuento frente a comparables,Chi-cuadrado independencia,54.8050,0.0479,0.0479,Se rechaza H0,0.0927,Cramers V,6381,40
6,7. Prima por dotacion de amenities,Kruskal-Wallis,206.0075,0.0000,0.0000,Se rechaza H0,0.0282,epsilon^2,7245,3
7,8. Accesibilidad y precio por m2,Kruskal-Wallis,169.8342,0.0000,0.0000,Se rechaza H0,0.0230,epsilon^2,7245,4
8,9. Distancia al subte,Kruskal-Wallis,148.2913,0.0000,0.0000,Se rechaza H0,0.0199,epsilon^2,7245,5
9,10. Indice de oportunidad de flip,Kruskal-Wallis,868.6581,0.0000,0.0000,Se rechaza H0,0.1193,epsilon^2,7245,6


## Tablas de apoyo para interpretar direccion del efecto

Los tests indican si hay evidencia estadistica; estas tablas muestran la direccion del patron.

In [6]:
display(
    work.groupby("Dotacion_amenities", observed=False)
    .agg(propiedades=("Precio_m2", "size"), mediana_precio_m2=("Precio_m2", "median"), promedio_amenities=("Cantidad_amenities", "mean"))
    .round(2)
)

display(
    work.groupby("Rango_accesibilidad", observed=False)
    .agg(propiedades=("Precio_m2", "size"), mediana_precio_m2=("Precio_m2", "median"), indice_accesibilidad_prom=("Indice_accesibilidad", "mean"))
    .round(2)
)

display(
    work.groupby("Rango_dist_subte", observed=False)
    .agg(propiedades=("Precio_m2", "size"), mediana_precio_m2=("Precio_m2", "median"), distancia_subte_mediana=(c("Dist_Subte_m"), "median"))
    .round(2)
)

,propiedades,mediana_precio_m2,promedio_amenities
Dotacion_amenities,,,
Baja,2415,"2,030.1900",0.7200
Media,2415,"2,120.0000",2.9600
Alta,2415,"2,314.8100",5.6800


,propiedades,mediana_precio_m2,indice_accesibilidad_prom
Rango_accesibilidad,,,
Baja,1812,"2,217.2000",0.4900
Media-baja,1811,"2,261.3600",0.6500
Media-alta,1811,"2,187.5000",0.7300
Alta,1811,"1,962.9600",0.8100


,propiedades,mediana_precio_m2,distancia_subte_mediana
Rango_dist_subte,,,
0-250 m,995,"1,973.6800",173.7500
250-500 m,2117,"2,124.3000",364.7000
500-750 m,1152,"2,230.6100",615.0400
750-1000 m,785,"2,404.5800",868.1800
Mas de 1000 m,2196,"2,142.8600","1,939.5000"


In [7]:
display(
    work.groupby(c("Cluster"))
    .agg(
        propiedades=("Indice_oportunidad_flip", "size"),
        mediana_indice_oportunidad=("Indice_oportunidad_flip", "median"),
        mediana_precio_m2=("Precio_m2", "median"),
        pct_mejorable=("Estado_mejorable", "mean")
    )
    .assign(pct_mejorable=lambda x: x["pct_mejorable"] * 100)
    .round(2)
)

display(
    work[work["N_comparables"] >= 5]
    .groupby(c("Barrio"))
    .agg(
        propiedades=("Precio_m2", "size"),
        pct_descuento_fuerte=("Descuento_fuerte", "mean"),
        mediana_subvaluacion=("Subvaluacion_%", "median")
    )
    .assign(pct_descuento_fuerte=lambda x: x["pct_descuento_fuerte"] * 100)
    .query("propiedades >= 30")
    .sort_values("pct_descuento_fuerte", ascending=False)
    .head(12)
    .round(2)
)

,propiedades,mediana_indice_oportunidad,mediana_precio_m2,pct_mejorable
sintetica_Cluster,,,,
0,817,22.6700,"1,775.0000",17.5000
1,74,27.4800,"4,785.8200",1.3500
2,2116,34.7200,"2,277.7800",10.7300
3,1923,30.5000,"2,338.7100",11.1300
4,977,36.0200,"2,459.0200",15.3500
5,1338,25.6100,"1,700.0000",15.1000


,propiedades,pct_descuento_fuerte,mediana_subvaluacion
enriquecida_Barrio,,,
Villa Lugano,54,37.0400,0.0000
Mataderos,65,35.3800,0.0000
La Boca,79,31.6500,0.0000
Colegiales,276,31.5200,0.0000
Chacarita,85,30.5900,0.0000
Liniers,54,29.6300,0.0000
Retiro,149,28.8600,0.0000
Barracas,167,28.1400,0.0000
Velez Sarsfield,32,28.1200,0.0000


## Conclusiones

La siguiente tabla resume cada KPI/hipotesis del notebook 6, su test asociado y la lectura estadistica. La evidencia estadistica no implica causalidad: los datos son precios publicados y los tests son principalmente bivariados.

In [8]:
conclusiones = resultados[[
    "kpi_notebook_6", "H0", "H1", "test", "p_value_ajustado_holm",
    "decision", "tamano_efecto", "medida_efecto", "lectura"
]].copy()

conclusiones

,kpi_notebook_6,H0,H1,test,p_value_ajustado_holm,decision,tamano_efecto,medida_efecto,lectura
0,1. Precio de entrada por zona,Precio_m2 tiene distribucion similar entre bar...,Al menos un barrio difiere en Precio_m2,Kruskal-Wallis,0.0000,Se rechaza H0,0.3048,epsilon^2,Valida que el barrio es una dimension relevant...
1,2. Profundidad de mercado,La oferta se distribuye uniformemente entre ba...,La oferta no se distribuye uniformemente entre...,Chi-cuadrado bondad de ajuste,0.0000,Se rechaza H0,0.1338,w,Valida que la profundidad de mercado varia sig...
2,3. Stock mejorable por barrio,Barrio y estado mejorable son independientes,La proporcion de mejorables depende del barrio,Chi-cuadrado independencia,0.0000,Se rechaza H0,0.1293,Cramers V,Valida que el stock mejorable no aparece igual...
3,4. Amenities por segmento de precio,Cantidad de amenities y Precio_m2 no estan aso...,Cantidad de amenities y Precio_m2 estan asociados,Spearman,0.0000,Se rechaza H0,0.1923,|rho|,Valida si los amenities funcionan como atribut...
4,5. Accesibilidad urbana por barrio,El indice de accesibilidad es similar entre ba...,Al menos un barrio difiere en accesibilidad,Kruskal-Wallis,0.0000,Se rechaza H0,0.5740,epsilon^2,Valida que la accesibilidad es un rasgo territ...
5,6. Descuento frente a comparables,El descuento fuerte es independiente del barrio,La frecuencia de descuento fuerte depende del ...,Chi-cuadrado independencia,0.0479,Se rechaza H0,0.0927,Cramers V,Valida si las oportunidades relativas se conce...
6,7. Prima por dotacion de amenities,Precio_m2 es similar entre dotaciones de ameni...,Precio_m2 difiere segun dotacion de amenities,Kruskal-Wallis,0.0000,Se rechaza H0,0.0282,epsilon^2,Valida la existencia de prima de precio asocia...
7,8. Accesibilidad y precio por m2,Precio_m2 es similar entre rangos de accesibil...,Precio_m2 difiere segun rango de accesibilidad,Kruskal-Wallis,0.0000,Se rechaza H0,0.0230,epsilon^2,Valida si la accesibilidad segmenta valores de...
8,9. Distancia al subte,Precio_m2 es similar entre rangos de distancia...,Precio_m2 difiere segun distancia al subte,Kruskal-Wallis,0.0000,Se rechaza H0,0.0199,epsilon^2,Valida si la cercania/lejana al subte cambia l...
9,10. Indice de oportunidad de flip,El indice de oportunidad es similar entre clus...,El indice de oportunidad difiere entre clusters,Kruskal-Wallis,0.0000,Se rechaza H0,0.1193,epsilon^2,Valida si los clusters agregados en el limpio ...


## Limitaciones

- Se trabaja con precios publicados, no precios de cierre.
- No se observan dias en mercado, costos de obra, margen de negociacion ni precio final de reventa.
- Las pruebas no controlan simultaneamente todas las variables; para eso haria falta un modelo multivariado.
- Por lo tanto, estos resultados validan patrones estadisticos descriptivos, no causalidad ni ROI real.